In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/home/dongmin/userdata/dongmin/omr-data-preprocessing


/home/dongmin/userdata/dongmin/omr-data-preprocessing/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
import os
import shutil
from pathlib import Path
from collections import defaultdict
from itertools import groupby

import math
import random

import json
import csv
import strictyaml as syaml

from tqdm import tqdm
import matplotlib.pyplot as plt

import cv2
import numpy as np
import music21
import musicdiff as md
import converter21 as c21

import xml.etree.ElementTree as ET
from modules.lmxe.lmxe import linearize_lmxe, delinearize_lmxe
from modules.lmxe.lmxe.utils import split_tokens_by_key_token, find_nth_token_index
from modules.lmxe.lmxe.Linearizer import LMXELinearizer
from modules.lmxe.lmxe.Delinearizer import LMXEDelinearizer
from modules.lmxe.lmxe.LMXEFile import LMXEFile, LMXEMetadata, get_score_type
from modules.lmxe.lmxe.symbolic.MxlFile import MusicXMLFile
from modules.lmxe.lmxe.symbolic.part_to_score import parts_to_score
from modules.lmxe.lmxe.symbolic.debug_compare import compare_parts

from omrdp.utils import load_ossq_metadata, parse_log_file
from omrdp.utils.image import crop_white_space
from omrdp.utils.symbolic.musicxml import parse_annotated_errors, inspect_error

from IPython.core.magic import register_cell_magic
@register_cell_magic
def skip(line, cell):
  return

## Load metadata

In [3]:
dataset_dir = Path.home() / 'userdata' / 'open-score-string-quartets'
dataset_data_dir = dataset_dir / 'data'
dataset_score_dir = dataset_dir / 'scores'

dataset_dir.exists()

True

In [4]:
metadata = load_ossq_metadata(dataset_data_dir / 'scores.yaml')
len(metadata)

122

In [5]:
temp_dir = Path.cwd() / 'temp'
temp_dir.mkdir(exist_ok=True)

## Pre-filtering

### Find part_count > 4

In [8]:
exceptions = []

pbar = tqdm(list(metadata.items()))
for score_id, body in pbar:
  sqid = body['sqid']
  sq_dir = body['path']
  
  pbar.set_description(f'{sq_dir}')
  
  sq_dir = dataset_score_dir / sq_dir
  
  musicxml = sq_dir / f'{sqid}_cleaned.musicxml'
  musicxml = MusicXMLFile.load_musicxml(musicxml)
  musicxml.get_parts()
  part_count = len(musicxml.parts['multi'])
  if part_count > 4:
    exceptions.append((sqid, part_count, sq_dir))

Wolf,_Hugo/String_Quartet: 100%|██████████| 122/122 [00:24<00:00,  5.07it/s]                                                              


In [9]:
exceptions

[('sq7158117',
  5,
  PosixPath('/home/dongmin/userdata/open-score-string-quartets/scores/Janáček,_Leoš/String_Quartet_No.1_“Kreutzer_Sonata”'))]